# SIGEC - Entrenamiento de modelos (Colab)

Entrena los dos modelos reales que faltan:
1. **Neumonia** en radiografias (MobileNetV2) -> `ml/pneumonia/models/pneumonia.keras` (+ `.tflite`).
2. **Inasistencia** (Random Forest) -> `ml/no-show/models/no-show.joblib`.

Antes: *Entorno de ejecucion -> Cambiar tipo -> GPU (T4)*. Necesitas tu `kaggle.json` (kaggle.com -> Settings -> API -> Create New Token).

> Proyecto academico. El modelo de neumonia es apoyo a la decision, no un diagnostico.

## 1. Repositorio
Si el repo es privado, pon un token de GitHub (solo lectura) en `GH_TOKEN`; si es publico, dejalo vacio. No guardes el token en el notebook compartido.

In [ ]:
## 1. Codigo del proyecto (sin GitHub)
Sube `sigec-ml.zip` (solo scripts de entrenamiento, ~20 KB). Se crea en tu repo local con:

```bash
zip -r sigec-ml.zip ml/pneumonia/*.py ml/no-show/train.py services/ms7-inteligencia/ml/app -x '*__pycache__*'
```

import os, zipfile
from google.colab import files
up = files.upload()  # sigec-ml.zip
os.makedirs('/content/sigec', exist_ok=True)
zipfile.ZipFile('sigec-ml.zip').extractall('/content/sigec')
%cd /content/sigec
!ls ml services/ms7-inteligencia/ml/app

In [ ]:
from google.colab import files
up = files.upload()  # kaggle.json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.replace('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
!pip -q install kaggle scikit-learn pandas joblib
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 3. Inasistencia (Random Forest, minutos)
Descarga `joniarroba/noshowappointments`. Division por fecha; informe en `ml/no-show/reports/`.

In [ ]:
!python ml/no-show/train.py
!ls -la ml/no-show/models ml/no-show/reports

## 4. Neumonia (MobileNetV2)
Descarga ~1.2 GB. Con GPU tarda ~20-40 min. Para probar primero con pocas imagenes: `--subconjunto 200 --epocas 2 --epocas-ajuste 0`.

In [ ]:
!python ml/pneumonia/train.py
!ls -la ml/pneumonia/models ml/pneumonia/reports

### 4.1 TFLite para la app movil / PWA (opcional)

In [ ]:
!python ml/pneumonia/convertir.py --modelo ml/pneumonia/models/pneumonia.keras --tflite --cuantizar
!ls -la ml/pneumonia/models

## 5. Revisar metricas
En neumonia mira sobre todo la **sensibilidad**; en inasistencia, recall/F1/AUC.

In [ ]:
import glob, json
for f in sorted(glob.glob('ml/*/reports/**/*.json', recursive=True)):
    print('==', f)
    d = json.load(open(f))
    print(json.dumps(d.get('metricas', d), indent=1, ensure_ascii=False)[:1500])

## 6. Descargar
Descomprime `resultados.zip` en la **raiz** de tu repo local (las rutas ya coinciden). Los `.keras`, `.tflite` y `.joblib` estan en `.gitignore`.

In [ ]:
!zip -rq /content/resultados.zip ml/pneumonia/models ml/pneumonia/reports ml/no-show/models ml/no-show/reports
files.download('/content/resultados.zip')